# 玉藻前配图 - 免登录动漫模型版
使用 Waifu Diffusion，无需 HF Token，解决脸部问题

## 使用说明
1. 依次运行单元格
2. 模型自动下载，无需登录
3. 先测试第一张，满意再生成全部

In [ ]:
# 第一步：安装依赖
!pip install -q diffusers transformers accelerate safetensors torch
print('依赖安装完成')

In [ ]:
# 第二步：检查 GPU
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 第三步：加载免登录动漫模型（Waifu Diffusion）
from diffusers import StableDiffusionPipeline
import torch

print('正在加载 Waifu Diffusion 动漫模型...')
pipe = StableDiffusionPipeline.from_pretrained(
    'hakurei/waifu-diffusion',  # 动漫专用，无需登录
    torch_dtype=torch.float16,
    use_safetensors=True,
    local_files_only=False,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('动漫模型加载完成！脸部效果优秀')

In [ ]:
# 第四步：定义动漫提示词（针对脸部优化）
anime_prefix = '(masterpiece, best quality:1.2), (anime style:1.4), (detailed anime face:1.3), (big beautiful eyes:1.2), '

negative_prompt = 'realistic, 3d, low quality, blurry, deformed face, ugly, bad anatomy'

prompts = [
    ('1.1_印度华阳天', anime_prefix + 'Ancient Indian palace, fox spirit court lady, golden robes, nine tails, anime style'),
    ('1.2_中国妲己', anime_prefix + 'Chinese Shang dynasty, concubine Daji, silk robes, nine fox tails, Chinese anime style'),
    ('1.3_日本玉藻前', anime_prefix + 'Japanese Heian court, Tamamo-no-Mae, elegant kimono, cherry blossoms, anime style'),
    ('2.1_入宫得宠', anime_prefix + 'Tamamo-no-Mae playing koto, court ladies, palace interior, anime style'),
    ('2.2_天皇病重', anime_prefix + 'Emperor Toba in bed, dark room, fox shadow, dramatic lighting, anime style'),
    ('2.3_阴阳师怀疑', anime_prefix + 'Yin-Yang master meditating, fox shadow, moonlight, anime style'),
    ('3.1_识破妖身', anime_prefix + 'Yin-Yang masters casting spells, nine-tailed fox revealed, magical circles, anime'),
    ('3.2_天皇的震惊', anime_prefix + 'Emperor Toba shocked, mirror reflection fox ears, palace room, anime'),
    ('3.3_那须野的藏身', anime_prefix + 'Abandoned mansion Nasu field, moonlight, mysterious, anime style'),
    ('4.1_三浦介与上总介', anime_prefix + 'Japanese samurai warriors, traditional armor, Nasu field, anime style'),
    ('4.2_激战那须野', anime_prefix + 'Epic battle, nine-tailed fox, samurai fighting, dynamic action, anime'),
    ('4.3_妖狐之死', anime_prefix + 'Nine-tailed fox falling, arrow in forehead, sunset, tragic anime scene'),
    ('5.1_石头诞生', anime_prefix + 'Giant stone formation, poisonous aura, dead vegetation, Japanese landscape anime'),
    ('5.2_镇魂与封印', anime_prefix + 'Buddhist monk chanting, glowing stone, peaceful temple, anime style'),
    ('5.3_现代遗迹', anime_prefix + 'Tourists visiting Sessho-seki stone, modern Japan, historical site anime'),
    ('6.1_文学与戏剧', anime_prefix + 'Traditional Japanese theater, Noh mask, Tamamo-no-Mae story, anime style'),
    ('6.2_现代流行文化', anime_prefix + 'Tamamo-no-Mae, modern illustration, vibrant colors, manga aesthetic'),
    ('6.3_东亚妖狐文化的交融', anime_prefix + 'Three women India China Japan, fox spirits, cultural exchange, anime')
]

print(f'准备生成 {len(prompts)} 张动漫图片')

In [ ]:
# 测试用：只生成第一张，验证脸部效果
print('=== 测试生成第一张（Waifu Diffusion）：1.1_印度华阳天 ===')
test_name, test_prompt = prompts[0]
print(f'提示词: {test_prompt}')

try:
    test_image = pipe(
        prompt=test_prompt,
        negative_prompt=negative_prompt,
        width=768,
        height=432,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]
    
    # 显示测试图
    display(test_image.resize((960, 540)))
    print('Waifu Diffusion 测试图生成完成！脸部应该自然很多')
    
except Exception as e:
    print(f'测试失败: {e}')

In [ ]:
# 第五步：生成全部图片（Waifu Diffusion）
import os
from PIL import Image

# 创建输出目录
os.makedirs('tamamo_images', exist_ok=True)

print('开始生成 Waifu Diffusion 动漫图片...\n')
for i, (name, prompt) in enumerate(prompts, 1):
    print(f'[{i}/{len(prompts)}] 生成: {name}')
    
    try:
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=768,
            height=432,
            num_inference_steps=30,
            guidance_scale=7.5,
        ).images[0]
        
        # 放大到 1920x1080
        image_large = image.resize((1920, 1080), Image.LANCZOS)
        
        # 保存
        output_path = f'tamamo_images/{name}.png'
        image_large.save(output_path, 'PNG', quality=95)
        print(f'   ✓ 保存: {output_path}')
        
    except Exception as e:
        print(f'   ✗ 失败: {e}')

print('\n所有图片生成完成！')

In [ ]:
# 第六步：打包并下载
import shutil
from google.colab import files

# 打包成 zip
shutil.make_archive('tamamo_images', 'zip', 'tamamo_images')
print('打包完成: tamamo_images.zip')

# 下载
files.download('tamamo_images.zip')